# 06b · CircuitKIT cross-check of the refusal circuit (toolkit chaining)

Runs Lexsi Labs' **CircuitKIT** (EAP-IG, node level) on the matched calibration pairs and compares its ranking with the components our own attribution + exact patching found in 06. Uses the tested adapter `safetypulse.adapters.circuitkit` (attach the `safetypulse-pkg` dataset).

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · `safetypulse-pkg` · outputs of **03 matched**, **04**, **06 matched** · `sp-models` |
| **Reads** | `pairs.jsonl`, `token_sets.json`, `probe_bundle/` |
| **Writes** (`/kaggle/working/`) | `circuitkit/crosscheck.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [ ]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

In [ ]:
import json, subprocess, sys
import torch

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "circuitkit==0.2.0"], check=True)
# Kaggle unwraps a dataset's single top-level folder, so find the package by its files and copy it into a
# directory actually named `safetypulse` before importing it.
import shutil
src = next(p.parent for p in sp.find_inputs("cli.py")
           if (p.parent / "check.py").exists() and (p.parent / "adapters").is_dir())
shutil.copytree(src, "/tmp/sp_pkg/safetypulse", dirs_exist_ok=True)
sys.path.insert(0, "/tmp/sp_pkg")
from safetypulse import ProbeBundle
from safetypulse.adapters import circuitkit as ck

MODEL = os.environ.get("SP_MODEL", sp.DEFAULT_MODEL)
N_PAIRS = int(os.environ.get("SP_N_PAIRS", 32)) if not sp.SMOKE else 4   # EAP-IG on a 3.9B model fills 2 x T4
K = 20
OUT = sp.out_dir("circuitkit")
token_sets = sp.load_json(sp.find_input("token_sets.json"))[MODEL]
bundle = ProbeBundle.load(sp.find_input("probe_bundle.json").parent)
# EAP compares clean and corrupt runs position by position, so keep only pairs whose chat-formatted harmful
# and harmless prompts have the same token length (CircuitKIT warns that other pairs give unreliable scores).
# Raw prompts by default: with Tiny Aya's chat template every prompt is ~385 tokens, and CircuitKIT's EAP then
# runs out of memory on a T4 (full-sequence logits over a 262k vocabulary). SP_CK_TEMPLATE=on restores it.
TEMPLATE = os.environ.get("SP_CK_TEMPLATE", "off")
tok = sp.load_tokenizer(MODEL)
n_tok = lambda text: len(tok(sp.format_chat(tok, [text])[0] if TEMPLATE == "on" else text,
                             add_special_tokens=False).input_ids)
calib = [p for p in sp.load_jsonl(sp.find_input("pairs.jsonl")) if p["split"] == "calib"]
pairs = [p for p in calib if n_tok(p["harmful"]) == n_tok(p["harmless"])][:N_PAIRS]
print(f"{len(pairs)} length-matched calib pairs (of {len(calib)}); {len(bundle.components)} components in our "
      f"bundle; layer {bundle.layer}")

In [ ]:
# EAP-IG on this 3.9B model does not fit one T4 (OOM) and CircuitKIT's discovery does not support a model
# split over two GPUs (device mismatch), so plain EAP on one GPU, one pair at a time (SP_CK_ALGO overrides).
ALGO = os.environ.get("SP_CK_ALGO", "eap")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
with sp.gpu_timer() as t:
    tl = ck.require_circuitkit().load_model(sp.model_source(MODEL), dtype="float16", algorithm=ALGO)
    theirs = ck.discover(tl, pairs, token_sets, algorithm=ALGO, batch_size=1, chat_template_mode=TEMPLATE)
result = {"algorithm": ALGO, "chat_template_mode": TEMPLATE, "n_pairs": len(pairs), "gpu_seconds": t["seconds"],
          "circuitkit_version": ck.require_circuitkit().__version__,
          **ck.compare([c.id for c in bundle.components], theirs, K)}
sp.save_json({**result, "circuitkit_scores": theirs}, OUT / "crosscheck.json")
{k: v for k, v in result.items() if not isinstance(v, (list, dict))}